# ABC Vincent - clean path

This notebook keeps only the minimum path needed to understand the experiment.

The direction is:

1. Load Vincent's functions and ABC class.
2. Choose one MU284 example.
3. Build the Ppi reference design.
4. Start ABC from Vincent's warm start: `omega = 0`, `rho = 0.5`.
5. Run a short ABC loop and watch whether it improves over Ppi.

## Very simple meaning

`Ppi` gives us Vincent's reference design.

`CaDsd(omega, rho)` gives many possible designs.

`ABCAlgorithm` searches over `omega` and `rho`.

The ABC iteration is inside `abc.optimize(...)`. At each iteration it tries new `omega/rho` values, keeps better ones, and reports the best variance efficiency found so far.

In [7]:
from pathlib import Path
import contextlib
import importlib.util
import io

import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "simulations_abc" / "terminal_running" / "run_abc_mu284_terminal.py").exists():
            return path
    raise RuntimeError("Could not find project root.")


PROJECT_ROOT = find_project_root()
RUNNER_PATH = PROJECT_ROOT / "simulations_abc" / "terminal_running" / "run_abc_mu284_terminal.py"
DATA_PATH = PROJECT_ROOT / "simulations_abc" / "populations" / "real" / "MU284.csv"

spec = importlib.util.spec_from_file_location("vincent_runner", RUNNER_PATH)
runner = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()):
    spec.loader.exec_module(runner)

print("Loaded:", RUNNER_PATH)
print("Data:", DATA_PATH)

Loaded: /home/bardia/projects/graphical-sampling/simulations_abc/terminal_running/run_abc_mu284_terminal.py
Data: /home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv


## Settings

Keep this small while learning. Larger `MAX_ITERATIONS` means ABC searches longer.

In [8]:
Z_VAR = "ME84"
Y_VAR = "P85"
SIZE_VAR = "P75"
N_SAMPLE = 5

COLONY_SIZE = 8
MAX_ITERATIONS = 1000
RANDOM_SEED = 12345

INITIAL_OMEGA_VALUE = 0.0
INITIAL_RHO_VALUE = 0.5

## Prepare one case

We use one variable `z`, one variable `y`, and inclusion probabilities `pi`.

The sorting step is important because CaDsd expects probabilities in descending order internally.

In [9]:
df = pd.read_csv(DATA_PATH)
N = len(df)

y_raw = df[Y_VAR].to_numpy(dtype=float)
z_raw = df[Z_VAR].to_numpy(dtype=float)
x_raw = df[SIZE_VAR].to_numpy(dtype=float)

pi_raw = runner.inclusionprobabilities(x_raw, N_SAMPLE)

# Same preparation style as the original notebook.
sort_idx = np.argsort(z_raw / pi_raw)
y_sorted = y_raw[sort_idx]
z_sorted = z_raw[sort_idx]
pi_sorted = pi_raw[sort_idx]

var_srs_y = N**2 * (1.0 - N_SAMPLE / N) * np.var(y_raw, ddof=1) / N_SAMPLE
var_srs_z = N**2 * (1.0 - N_SAMPLE / N) * np.var(z_raw, ddof=1) / N_SAMPLE

print(f"N = {N}")
print(f"sample size = {N_SAMPLE}")
print(f"Corr({Y_VAR}, {Z_VAR}) = {np.corrcoef(y_raw, z_raw)[0, 1]:.3f}")
print(f"sum(pi) = {pi_raw.sum():.6f}")

N = 281
sample size = 5
Corr(P85, ME84) = 0.988
sum(pi) = 5.000000


## Build the ABC object

This computes the Ppi reference immediately.

The warm start is the key Vincent idea here: `omega = 0` gives approximately the Ppi variance, for any fixed `rho`.

In [10]:
abc = runner.ABCAlgorithm(
    y_sorted=y_sorted,
    z_sorted=z_sorted,
    pik_sorted=pi_sorted,
    var_srs_y=var_srs_y,
    var_srs_z=var_srs_z,
    M=N_SAMPLE,
    n=N_SAMPLE,
    case_name=f"{Z_VAR}_n{N_SAMPLE}_clean",
    objective="eff_z",
    enforce_cadsd_order=True,
    random_state=RANDOM_SEED,
    validation_mode="fast",
    initial_omega_value=INITIAL_OMEGA_VALUE,
    initial_rho_value=INITIAL_RHO_VALUE,
)

print("Ppi reference efficiency:")
print(f"  z: {abc.eff_z_optimal:.4f}")
print(f"  y: {abc.eff_y_optimal:.4f}")

Ppi reference efficiency:
  z: 57.7909
  y: 127.2314


## Where does the ABC iteration run?

It runs in this call:

`abc.optimize(max_iterations=MAX_ITERATIONS, ...)`

Inside that method, the loop is conceptually:

```text
for iteration in range(max_iterations):
    employed bees try new omega/rho
    onlooker bees try promising omega/rho
    scout bees replace stuck solutions
    local search makes small moves near the best solution
    keep the best solution found
```

So the direction of travel is from the Ppi warm start toward any CaDsd design with better variance.

In [ ]:
result = abc.optimize(
    colony_size=COLONY_SIZE,
    max_iterations=MAX_ITERATIONS,
    limit=8,
    verbose=True,
    progress_interval=1,
    local_search_interval=5,
    local_search_attempts=1,
    onlooker_factor=0.5,
    early_stopping=False,
    min_iterations=1,
)

 ABC ALGORITHM - ME84_n5_clean
 Configuration:
   colony_size          = 8
   max_iterations       = 1000
   abandonment limit    = 8
   objective            = eff_z
   warm start           = omega=0, rho=0.5
   validation mode      = fast
   onlooker factor      = 0.5
   early stopping       = False, patience=250
   time limit           = None
   random search        = OFF
   progress interval    = every 1 iterations
   local search interval= every 5 iterations
 Reference Pπ efficiency versus SRS:
   Pπ_z = 57.79
   Pπ_y = 127.23

 Initializing 8 food sources...

    Initialized 8 food sources (requested 8)
    Warm start: omega=0, rho=0.5
    Best initial: eff_z=57.7818, eff_y=127.1711
 iter | ABC_z/Pπ_z | ABC_y/Pπ_y | scout | ABC val% |  elapsed
-------------------------------------------------------------
    1 |       1.00 |       0.99 |     0 |  100.00% |    2.66s
    2 |       1.00 |       0.99 |     0 |  100.00% |    3.98s
    3 |       1.00 |       0.99 |     0 |  100.00% |   

## Read the result

`ABC_z / Ppi_z = 1.00` means equal to Ppi.

Above `1.00` means ABC found something better than Ppi for `z`.

Below `1.00` means it did not beat Ppi.

In [ ]:
summary = {
    "Ppi_z_eff": result["optimal_eff_z"],
    "ABC_z_eff": result["best_eff_z"],
    "ABC_z_over_Ppi_z": result.get("abc_z_over_Ppi_z", result["best_eff_z"] / result["optimal_eff_z"]),
    "Ppi_y_eff": result["optimal_eff_y"],
    "ABC_y_eff": result["best_eff_y"],
    "ABC_y_over_Ppi_y": result.get("abc_y_over_Ppi_y", result["best_eff_y"] / result["optimal_eff_y"]),
    "evaluations": result["eval_count"],
    "valid_evaluations": result["valid_count"],
}

pd.Series(summary)

Ppi_z_eff             57.790942
ABC_z_eff             59.211075
ABC_z_over_Ppi_z       1.024574
Ppi_y_eff            127.231390
ABC_y_eff            128.442136
ABC_y_over_Ppi_y       1.009516
evaluations          180.000000
valid_evaluations    180.000000
dtype: float64

## Why the old notebook is crowded

The old `ABC_Vincent.ipynb` mixes several jobs in one place:

- Python translation of Vincent's R functions.
- ABC algorithm development.
- MU284 data experiments.
- Reciprocal CaDsd checks.
- Warm-start checks.
- Sensitivity analysis.
- Exporting CSV files for R/Python exchange.

That is useful as a laboratory notebook, but it is hard to read. This clean notebook is only for understanding the main direction.